# Exercises: k-Nearest Neighbours

**Practical Machine Learning** · Sayan CHAKI · LIRIS, Université Lyon 2

Three exercises:

| # | Topic | Main skills |
|---|---|---|
| 1 | **k-NN from scratch** on iris (2 features) | broadcasting, majority vote, decision boundaries, bias and variance |
| 2 | **Breast cancer** diagnosis | scaling, leakage-free pipelines, tuning `k`, the one-standard-error rule, decision thresholds |
| 3 | **The curse of dimensionality** | distance concentration, irrelevant features |

**Open in Colab.** In [colab.research.google.com](https://colab.research.google.com) choose *File → Upload notebook* (or open it from Google Drive). Everything uses datasets shipped with scikit-learn, so no download or upload of data is needed.

**Rules of the game**
* Keep all `random_state` / seeds as given, so results are comparable across the class.
* Never use the test set to choose a model or a hyperparameter; it is opened **once**, at the end.
* Cells marked `# TODO` are yours. Cells marked `# CHECK` test your work: run them, they must pass.
* Questions marked ✍️ need a short written answer (2 to 4 sentences, with numbers from your results).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     cross_val_predict, GridSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             recall_score, precision_score, euclidean_distances)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---
# Exercise 1 · k-NN from scratch

**Context.** The iris dataset: 150 flowers, 3 species. We keep only the two **sepal** measurements, so that we can draw the decision regions and so that the classes overlap (the petal measurements would make the problem too easy).

### 1.1 Data
Run the cell: it splits the data (70 % train, 30 % test, stratified) and plots the training points.

In [ ]:
iris = load_iris()
X = iris.data[:, :2]          # sepal length, sepal width (cm)
y = iris.target               # 0 setosa, 1 versicolor, 2 virginica
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0)
print("train:", X_train.shape, " test:", X_test.shape)

fig, ax = plt.subplots()
for c, name in enumerate(iris.target_names):
    ax.scatter(*X_train[y_train == c].T, label=name, s=25, alpha=0.8)
ax.set_xlabel("sepal length (cm)"); ax.set_ylabel("sepal width (cm)"); ax.legend()
plt.show()

### 1.2 All pairwise distances at once
Write `pairwise_sq_dists(A, B)` returning the matrix $D$ with $D_{ij}=\lVert A_i-B_j\rVert^2$, of shape `(len(A), len(B))`, **without Python loops**.

*Hint:* `A[:, None, :] - B[None, :, :]` has shape `(len(A), len(B), n_features)`. An alternative uses $\lVert a-b\rVert^2=\lVert a\rVert^2+\lVert b\rVert^2-2\,a\cdot b$.

In [ ]:
# TODO
def pairwise_sq_dists(A, B):
    """D[i, j] = squared Euclidean distance between A[i] and B[j]."""
    raise NotImplementedError("TODO: write pairwise_sq_dists")

In [ ]:
# CHECK (run this cell, it must pass)
A = np.array([[0., 0.], [1., 1.]])
B = np.array([[3., 4.], [1., 0.], [0., 0.]])
D = pairwise_sq_dists(A, B)
assert D.shape == (2, 3), "wrong shape"
assert np.allclose(D, [[25, 1, 0], [13, 1, 2]]), "wrong values on the toy example"
assert np.allclose(pairwise_sq_dists(X_test, X_train), euclidean_distances(X_test, X_train) ** 2)
print("✅ pairwise_sq_dists is correct")

### 1.3 The classifier
Write `knn_predict(X_train, y_train, X_query, k)`:
1. compute the distances from each query point to every training point;
2. take the indices of the `k` smallest (`np.argsort(..., axis=1, kind="stable")`);
3. count the votes per class (`np.bincount` with `minlength`) and return the class with most votes. If two classes tie, `argmax` returns the smaller label; keep that rule.

In [ ]:
# TODO
def knn_predict(X_train, y_train, X_query, k):
    raise NotImplementedError("TODO: write knn_predict")

In [ ]:
# CHECK (run this cell, it must pass)
# 1) exact test on a jittered copy of the data, where no two distances are equal
jit = np.random.default_rng(1)
Xj_tr = X_train + jit.normal(scale=1e-4, size=X_train.shape)
Xj_te = X_test + jit.normal(scale=1e-4, size=X_test.shape)
for k in (1, 5, 15):
    ref = KNeighborsClassifier(n_neighbors=k, algorithm="brute").fit(Xj_tr, y_train).predict(Xj_te)
    assert (knn_predict(Xj_tr, y_train, Xj_te, k) == ref).all(), f"k={k}: predictions differ from scikit-learn"
# 2) on the real (rounded) data, ties between equal distances may be broken differently
for k in (1, 5, 15):
    mine = knn_predict(X_train, y_train, X_test, k)
    ref = KNeighborsClassifier(n_neighbors=k, algorithm="brute").fit(X_train, y_train).predict(X_test)
    print(f"k={k:2d}  agreement with scikit-learn: {(mine == ref).mean():.3f}   test accuracy: {accuracy_score(y_test, mine):.3f}")
print("✅ knn_predict agrees with scikit-learn")

> The iris measurements are rounded to 0.1 cm, so several training points are at exactly the same distance from a query. Different tie-breaking rules can then pick different neighbours, which explains why the agreement on the real data can be below 100 % (the CHECK therefore also tests on a copy with tiny random jitter, where there are no ties).

### 1.4 Decision regions
The helper below colours every point of a grid by its predicted class. Draw the regions of **your** classifier for `k = 1`, `k = 15` and `k = 60`.

In [ ]:
def plot_regions(predict, ax, title):
    xx, yy = np.meshgrid(np.linspace(4, 8.2, 220), np.linspace(1.8, 4.6, 160))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, levels=[-0.5, 0.5, 1.5, 2.5], cmap="viridis")
    ax.scatter(*X_train.T, c=y_train, s=15, cmap="viridis", edgecolor="k", linewidth=0.3)
    ax.set_title(title); ax.set_xlabel("sepal length"); ax.set_ylabel("sepal width")

In [ ]:
# TODO
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, k in zip(axes, [1, 15, 60]):
    ...  # call plot_regions with a lambda that uses knn_predict
plt.tight_layout(); plt.show()

### 1.5 Training and test accuracy as a function of k
For every `k` from 1 to 60, fit `KNeighborsClassifier(n_neighbors=k)` (scikit-learn is fine here) and store the accuracy on the training set and on the test set in two lists `train_acc` and `test_acc`. Plot both curves.

In [ ]:
# TODO
ks = np.arange(1, 61)
train_acc, test_acc = [], []
...

In [ ]:
# CHECK (run this cell, it must pass)
assert len(train_acc) == 60 and len(test_acc) == 60
assert train_acc[0] > train_acc[-1], "training accuracy should drop as k grows"
print("✅ curves computed")

### ✍️ Question 1
(a) Describe the three decision-region plots of 1.4: which one overfits, which one underfits, and how can you see it?
(b) Training accuracy at `k = 1` is **not** 100 %. Why? (Look at the data: the cell above counts duplicated points.)
(c) A classmate picks the `k` with the best **test** accuracy in 1.5 and reports that accuracy. What is wrong with this, and what should be done instead?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

---
# Exercise 2 · Breast cancer diagnosis

**Context.** 569 tumours described by 30 measurements computed from a digitised image of a fine-needle aspirate (radius, texture, area, smoothness, ...). Target: **0 = malignant, 1 = benign**. Careful with this coding: in medicine the "positive" class of interest is the malignant one, here labelled 0.

### 2.1 Split and look at the scales

In [ ]:
data = load_breast_cancer(as_frame=True)
Xbc, ybc = data.data, data.target
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    Xbc, ybc, test_size=0.25, stratify=ybc, random_state=0)
print("train:", Xb_train.shape, " test:", Xb_test.shape)
print("class proportions (train):", yb_train.value_counts(normalize=True).round(3).to_dict())
Xb_train.describe().T[["mean", "std", "min", "max"]].sort_values("std").iloc[[0, 1, 2, -3, -2, -1]]

### 2.2 Does scaling matter?
Using `cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)` on the **training set only**, compute the mean cross-validated accuracy of
* `KNeighborsClassifier(n_neighbors=5)` on the raw features → `acc_raw`
* a pipeline `StandardScaler` → `KNeighborsClassifier(n_neighbors=5)` → `acc_scaled`

In [ ]:
# TODO
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
acc_raw = None
acc_scaled = None

In [ ]:
# CHECK (run this cell, it must pass)
assert acc_raw is not None and acc_scaled is not None
assert 0.8 < acc_raw < 1 and 0.8 < acc_scaled < 1
assert acc_scaled > acc_raw, "scaling should help here"
print("✅ scaling comparison done")

### ✍️ Question 2
Using the table printed in 2.1, explain why the raw-feature k-NN is worse. Which features dominate the distance, and why would scaling the whole dataset **before** splitting into folds be a mistake?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

### 2.3 Tuning k and the weighting scheme
Use `GridSearchCV` on the scaled pipeline with
* `n_neighbors` in `1, 3, 5, ..., 41`,
* `weights` in `["uniform", "distance"]`,

with the same `cv`. Store the fitted search in `search`. Then plot the mean CV accuracy as a function of `k` for both weighting schemes, with error bars equal to the **standard error** (`std / sqrt(5)`).

In [ ]:
# TODO
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
param_grid = {}   # fill in; step names are lowercase class names, e.g. "kneighborsclassifier__n_neighbors"
search = None

In [ ]:
# CHECK (run this cell, it must pass)
assert search is not None
assert search.best_params_["kneighborsclassifier__n_neighbors"] in range(1, 42, 2)
assert search.best_score_ > 0.95
print("✅ grid search done")

### 2.4 The one-standard-error rule
Among all `uniform` models whose mean CV accuracy is within **one standard error** of the best `uniform` model, choose the **simplest**. For k-NN, which `k` is "simplest"? Store your choice in `k_1se`.

In [ ]:
# TODO
k_1se = None

In [ ]:
# CHECK (run this cell, it must pass)
assert k_1se is not None and k_1se % 2 == 1
u_ = res[res["w"] == "uniform"]
assert k_1se >= int(u_.loc[u_["mean_test_score"].idxmax(), "k"]), "the simplest k-NN model has the LARGEST k"
print("✅ one-SE choice made")

### ✍️ Question 3
Why is the *largest* admissible `k` the simplest model? How far is the one-SE choice from the best CV accuracy, and why is that difference not worth chasing?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

### 2.5 Open the test set, once
Fit the pipeline with `k = k_1se` and uniform weights on the whole training set. Report the test accuracy (`test_acc_knn`), the accuracy of a `DummyClassifier()` baseline (`test_acc_dummy`) and draw the confusion matrix.

In [ ]:
# TODO
final = None
test_acc_knn = None
test_acc_dummy = None

In [ ]:
# CHECK (run this cell, it must pass)
assert test_acc_knn > test_acc_dummy + 0.25
print("✅ test evaluation done")

### 2.6 Moving the decision threshold (training set only)
Missing a malignant tumour is much worse than a false alarm. `predict_proba` returns, for each tumour, the fraction of its `k` neighbours that are malignant (column 0).

Using `cross_val_predict(..., method="predict_proba")` on the **training set**, compute the recall and precision **of the malignant class** when a tumour is declared malignant if its malignant probability is ≥ `t`, for `t` in `[0.5, 0.3, 0.2, 0.1]`. Store them in a DataFrame `thr_table` with columns `t`, `recall_malignant`, `precision_malignant`.

*Hint:* `recall_score(y_true, y_pred, pos_label=0)`.

In [ ]:
# TODO
thr_table = None

In [ ]:
# CHECK (run this cell, it must pass)
assert list(thr_table.columns) == ["t", "recall_malignant", "precision_malignant"]
assert thr_table["recall_malignant"].is_monotonic_increasing, "lower threshold => more tumours flagged => recall cannot decrease"
print("✅ threshold table done")

### ✍️ Question 4
Which threshold would you recommend for a screening tool, and what is the price? Why must this choice be made on cross-validated predictions of the training set rather than on the test set?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

---
# Exercise 3 · The curse of dimensionality

### 3.1 Distances concentrate
Write `distance_contrast(d, n=500, seed=0)`: draw `n` points uniformly in $[0,1]^d$ and one query point, compute the Euclidean distances from the query to the `n` points and return the **relative contrast** $\dfrac{d_{\max}-d_{\min}}{d_{\min}}$.

Compute it for `dims = [1, 2, 5, 10, 50, 100, 500, 1000]` into a list `contrasts` and plot it on log-log axes.

In [ ]:
# TODO
def distance_contrast(d, n=500, seed=0):
    rng = np.random.default_rng(seed)
    raise NotImplementedError("TODO")

dims = [1, 2, 5, 10, 50, 100, 500, 1000]
contrasts = []

In [ ]:
# CHECK (run this cell, it must pass)
assert len(contrasts) == len(dims)
assert contrasts[0] > 100 * contrasts[-1], "contrast should collapse in high dimension"
print("✅ distance concentration observed")

### 3.2 Irrelevant features
Back to the breast cancer training set. Append `m` columns of pure Gaussian noise (use `rng = np.random.default_rng(0)`) for `m` in `[0, 10, 50, 200, 1000]`, and compute the mean CV accuracy (same `cv`) of
* the scaled k-NN pipeline with `k = k_1se` → list `acc_knn_noise`
* a scaled `LogisticRegression(max_iter=5000)` → list `acc_lr_noise`

In [ ]:
# TODO
ms = [0, 10, 50, 200, 1000]
acc_knn_noise, acc_lr_noise = [], []

In [ ]:
# CHECK (run this cell, it must pass)
assert len(acc_knn_noise) == 5 and len(acc_lr_noise) == 5
assert acc_knn_noise[0] - acc_knn_noise[-1] > 0.05, "k-NN should suffer from 1000 noise features"
print("✅ noise experiment done")

### ✍️ Question 5
Relate 3.1 and 3.2. Why does k-NN degrade so much faster than logistic regression when noise features are added? Name two practical remedies.

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*